In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/reemalharthii/train-videos/train_videos.csv
/kaggle/input/datasets/reemalharthii/test-videos/test_videos.csv
/kaggle/input/datasets/reemalharthii/sample-submission/sample_submission.csv


**1: Imports and Setup**

In [4]:
import warnings
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error

warnings.filterwarnings('ignore')
print("Libraries imported successfully.")

Libraries imported successfully.


**2: Load Datasets**

In [7]:
import os

# Automatically locate and load the datasets from Kaggle input path
input_dir = "/kaggle/input"
train_path = None
test_path = None

for root, dirs, files in os.walk(input_dir):
    if "train_videos.csv" in files:
        train_path = os.path.join(root, "train_videos.csv")
    if "test_videos.csv" in files:
        test_path = os.path.join(root, "test_videos.csv")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"Train Shape: {train_df.shape}")
print(f"Test Shape: {test_df.shape}")

Train Shape: (12000, 27)
Test Shape: (3001, 26)


**3: Feature Engineering Function**

In [9]:
def extract_engagement_features(df):
    """
    Creates interaction ratios, cumulative sums, and growth statistics 
    from historical time-series data (Days 0 to 5).
    """
    df = df.copy()
    
    # Identify daily engagement columns dynamically
    view_cols = [c for c in df.columns if 'view' in c.lower() and 'target' not in c.lower()]
    like_cols = [c for c in df.columns if 'like' in c.lower()]
    share_cols = [c for c in df.columns if 'share' in c.lower()]
    comment_cols = [c for c in df.columns if 'comment' in c.lower()]

    # Aggregations across Days 0-5
    if view_cols:
        df['total_views_0_5'] = df[view_cols].sum(axis=1)
        df['mean_views_0_5'] = df[view_cols].mean(axis=1)
        df['std_views_0_5'] = df[view_cols].std(axis=1)
        df['max_views_0_5'] = df[view_cols].max(axis=1)

    # Interaction Ratios
    if like_cols and view_cols:
        df['total_likes_0_5'] = df[like_cols].sum(axis=1)
        df['overall_like_ratio'] = df['total_likes_0_5'] / (df['total_views_0_5'] + 1)

    if share_cols and view_cols:
        df['total_shares_0_5'] = df[share_cols].sum(axis=1)
        df['overall_share_ratio'] = df['total_shares_0_5'] / (df['total_views_0_5'] + 1)

    if comment_cols and view_cols:
        df['total_comments_0_5'] = df[comment_cols].sum(axis=1)
        df['overall_comment_ratio'] = df['total_comments_0_5'] / (df['total_views_0_5'] + 1)

    # Growth rate from Day 0 to Day 5
    if len(view_cols) >= 2:
        df['views_growth_rate'] = (df[view_cols[-1]] - df[view_cols[0]]) / (df[view_cols[0]] + 1)

    # Convert categorical object columns
    cat_cols = df.select_dtypes(include=['object']).columns
    for col in cat_cols:
        df[col] = df[col].astype('category')

    return df

# Apply feature engineering
train_processed = extract_engagement_features(train_df)
test_processed = extract_engagement_features(test_df)

**4: Prepare Data & Target Transformation**

In [10]:
# Automatically identify target and ID columns
target_col = [c for c in train_df.columns if 'target' in c.lower() or '30' in c][0]
id_col = 'id' if 'id' in train_df.columns else train_df.columns[0]

features = [c for c in train_processed.columns if c not in [id_col, target_col]]

X = train_processed[features]
# Log-transform target to handle skewed Power-Law distribution and stabilize RMSE
y = np.log1p(np.maximum(0, train_processed[target_col]))

X_test = test_processed[features]

print(f"Features count: {len(features)}")
print(f"Target Column: {target_col}")

Features count: 25
Target Column: target_day30_views


**5: Cross-Validation & Model Training (LightGBM)**

In [11]:
# 5-Fold Cross-Validation Setup
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(train_processed))
test_preds = np.zeros(len(test_processed))

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    model = LGBMRegressor(
        n_estimators=2000,
        learning_rate=0.03,
        num_leaves=31,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[]
    )

    # Predictions in log space
    val_pred = model.predict(X_val)
    oof_preds[val_idx] = val_pred
    
    # Accumulate averaged test predictions
    test_preds += model.predict(X_test) / kf.n_splits

# Convert OOF predictions back to original scale
final_oof_true = train_processed[target_col].values
final_oof_pred = np.expm1(np.maximum(0, oof_preds))

# Calculate Out-Of-Fold RMSE score
cv_rmse = root_mean_squared_error(final_oof_true, final_oof_pred)
print(f"--------------------------------------")
print(f"Overall Out-Of-Fold RMSE: {cv_rmse:.4f}")
print(f"--------------------------------------")

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002763 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4054
[LightGBM] [Info] Number of data points in the train set: 9600, number of used features: 24
[LightGBM] [Info] Start training from score 7.129470
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001410 seconds.
You can set `force_col_wise=t

**6: Generate Submission CSV File**

In [12]:
# Convert test predictions back from log space
final_test_preds = np.expm1(np.maximum(0, test_preds))

# Create submission DataFrame
submission = pd.DataFrame({
    id_col: test_processed[id_col],
    target_col: final_test_preds
})

# Save to CSV
submission.to_csv('submission.csv', index=False)
print("Submission file successfully saved as 'submission.csv'")

Submission file successfully saved as 'submission.csv'
